# 01 数据清洗

加载 50 万条客户数据，检查数据质量，处理负值异常。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv('../customerData_500k.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# 缺失值与重复检查
print(f'Missing values:\n{df.isnull().sum()}')
print(f'\nDuplicates: {df.duplicated().sum()}')

In [ ]:
# 检查数值字段的异常值（负值）
num_cols = ['NumberOfPurchases','TimeSpentOnWebsite','CustomerTenureYears','LastPurchaseDaysAgo']
for col in num_cols:
    neg = (df[col] < 0).sum()
    print(f'{col}: min={df[col].min():.2f}  max={df[col].max():.2f}  negative={neg}')

In [ ]:
# 清洗
# 1) 负值 -> 0（购买次数、时长、年限不能为负）
for col in num_cols:
    df.loc[df[col] < 0, col] = 0

# 2) 移除年龄 < 17 的记录（有独立收入但未成年的矛盾数据）
df = df[df['Age'] >= 17]

print(f'清洗后: {df.shape}')
print(f'购买率: {df["PurchaseStatus"].mean()*100:.1f}%')

In [ ]:
# 分类字段分布
cat_cols = ['Gender','ProductCategory','PreferredDevice','Region','ReferralSource','CustomerSegment']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, cat_cols):
    dist = df[col].value_counts()
    ax.pie(dist.values, labels=dist.index, autopct='%1.1f%%', startangle=90)
    ax.set_title(col, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 清洗小结

- 原始 500,000 条，清洗后 499,381 条
- 修复负值：NumberOfPurchases(394)、TimeSpentOnWebsite(8,472)、CustomerTenureYears(11,145)、LastPurchaseDaysAgo(7,907)
- 移除 619 条年龄 < 17
- 零缺失值、零重复

> **注意**：本 notebook 展示清洗探索过程。后续 02/03 通过 `utils.data_loader.load_cleaned_data()` 统一加载清洗后数据——清洗规则的唯一定义在 `utils/data_loader.py`，修改清洗逻辑时只需改那一个文件。

下一步 → `02_exploratory_analysis.ipynb`